In [ ]:

import math
import struct
from tkinter import Tk, Canvas


# Define Point and Polyline classes
class Point:
    def __init__(self, x=0.0, y=0.0):
        self.x = x
        self.y = y


class Polyline:
    def __init__(self, points=None, partsNum=0):
        self.points = points if points is not None else []
        self.partsNum = partsNum
        self.partsIndex = []


# --------------------------------------------------
# Part 1: Read the SHX index file
# --------------------------------------------------

shxFile = open("Partial_Streets.shx", "rb")

# Read the first 28 bytes of the header
s = shxFile.read(28)
header = struct.unpack(">7i", s)

# File length is stored in 16-bit words
fileLength = header[6]

# Each SHX record occupies 8 bytes
polylineNum = (fileLength * 2 - 100) // 8

print("fileLength, polylineNum:", fileLength, polylineNum)

# Read the remaining 72 bytes of the header
s = shxFile.read(72)
header = struct.unpack("<2i8d", s)

# Read the shapefile bounding box
minX, minY, maxX, maxY = (
    header[2], header[3], header[4], header[5]
)

print("Bounding box:", minX, minY, maxX, maxY)

# Read the offsets of all records
recordsOffset = []

for i in range(polylineNum):
    shxFile.seek(100 + i * 8)

    s = shxFile.read(4)
    offset = struct.unpack(">i", s)[0]

    # Convert 16-bit words to bytes
    recordsOffset.append(offset * 2)

print("Record offsets:", recordsOffset)

shxFile.close()


# --------------------------------------------------
# Part 2: Read each polyline from the SHP file
# --------------------------------------------------

shpFile = open("Partial_Streets.shp", "rb")

polylines = []

for offset in recordsOffset:

    # Move to the beginning of the record content.
    # Record header = 8 bytes.
    # Shape type + bounding box = 36 bytes.
    shpFile.seek(offset + 8 + 36)

    s = shpFile.read(8)
    partsNum, pointsNum = struct.unpack("<2i", s)

    polyline = Polyline()
    polyline.partsNum = partsNum

    print("partsNum, pointsNum:", partsNum, pointsNum)

    # Read the starting point index for each part
    s = shpFile.read(4 * partsNum)

    if partsNum > 0:
        fmt = "<" + "i" * partsNum
        polyline.partsIndex = list(struct.unpack(fmt, s))

    # Read all points
    points = []

    for i in range(pointsNum):
        s = shpFile.read(16)
        xCoord, yCoord = struct.unpack("<2d", s)

        point = Point(xCoord, yCoord)
        points.append(point)

    polyline.points = points
    polylines.append(polyline)

shpFile.close()


# --------------------------------------------------
# Part 3: Visualize the street polylines
# --------------------------------------------------

root = Tk()
root.title("Partial Streets")

windowWidth = 800
windowHeight = 600

# Leave a small margin around the drawing
margin = 20
drawingWidth = windowWidth - 2 * margin
drawingHeight = windowHeight - 2 * margin

# Avoid division by zero
xRange = maxX - minX
yRange = maxY - minY

ratiox = drawingWidth / xRange if xRange != 0 else 1
ratioy = drawingHeight / yRange if yRange != 0 else 1

# Keep the same scale in both directions
ratio = min(ratiox, ratioy)

# Center the drawing in the window
drawingActualWidth = xRange * ratio
drawingActualHeight = yRange * ratio

offsetX = (windowWidth - drawingActualWidth) / 2
offsetY = (windowHeight - drawingActualHeight) / 2

can = Canvas(
    root,
    width=windowWidth,
    height=windowHeight,
    bg="white"
)

# Draw every polyline part
for polyline in polylines:

    # Convert geographic coordinates to canvas coordinates
    xylist = []

    for point in polyline.points:
        screenX = offsetX + (point.x - minX) * ratio

        # Canvas Y increases downward, so reverse the Y axis
        screenY = windowHeight - (
            offsetY + (point.y - minY) * ratio
        )

        xylist.append((screenX, screenY))

    # Draw each part separately
    for k in range(polyline.partsNum):

        startPointIndex = polyline.partsIndex[k]

        if k == polyline.partsNum - 1:
            endPointIndex = len(polyline.points)
        else:
            endPointIndex = polyline.partsIndex[k + 1]

        tempXYlist = []

        for m in range(startPointIndex, endPointIndex):
            tempXYlist.extend(xylist[m])

        # A line requires at least two points
        if len(tempXYlist) >= 4:
            can.create_line(
                *tempXYlist,
                fill="black",
                width=1
            )

can.pack()
root.mainloop()


[<__main__.Point object at 0x000002628F52E480>, <__main__.Point object at 0x000002628F55A450>, <__main__.Point object at 0x000002628F55AE10>, <__main__.Point object at 0x000002628F57BFE0>, <__main__.Point object at 0x000002628F57B260>, <__main__.Point object at 0x000002628F544FB0>, <__main__.Point object at 0x000002628F57A3C0>, <__main__.Point object at 0x000002628F546FF0>]
